# NLP Sentiment Analysis — Movie Reviews

**Author:** Tanaz Piriaei  
**M2 Informatique** — Données, Intelligence & Systèmes Intelligents  
Université Claude Bernard Lyon 1

---

This notebook compares multiple approaches for sentiment analysis on movie reviews:
- **Lexicon-based:** AFINN, VADER, WordNet
- **Deep Learning:** Word2Vec (CBOW + Skip-gram) + Neural Network, LSTM

## 1. Install & Import Libraries

In [ ]:
# Install required libraries
!pip install -q afinn contractions datasets

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter

# NLP
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('sentiwordnet', quiet=True)
nltk.download('vader_lexicon', quiet=True)

from nltk.tokenize.toktok import ToktokTokenizer
from nltk.corpus import stopwords, sentiwordnet as swn
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from bs4 import BeautifulSoup
from contractions import contractions_dict
from afinn import Afinn

# ML / DL
import gensim
from scipy.spatial import distance
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

print('All libraries imported successfully!')

## 2. Load Dataset (IMDB — public)

In [ ]:
from datasets import load_dataset

# Load public IMDB dataset (no CSV needed)
print('Loading IMDB dataset...')
dataset = load_dataset('imdb')

# Convert to DataFrame
train_raw = pd.DataFrame(dataset['train'])
test_raw  = pd.DataFrame(dataset['test'])
df = pd.concat([train_raw, test_raw], ignore_index=True)
df.columns = ['review', 'sentiment']

# Balance: 2500 positive + 2500 negative
pos = df[df['sentiment'] == 1].sample(2500, random_state=42)
neg = df[df['sentiment'] == 0].sample(2500, random_state=42)
df = pd.concat([pos, neg]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(df['sentiment'].value_counts())
df.head(3)

## 3. Text Preprocessing

In [ ]:
tokenizer = ToktokTokenizer()
stopword_list = set(stopwords.words('english'))

def strip_html(text):
    return BeautifulSoup(text, 'html.parser').get_text()

def remove_accented_chars(text):
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

def expand_contractions(text, cd=contractions_dict):
    pattern = re.compile(f"({'|'.join(cd.keys())})", flags=re.IGNORECASE)
    return pattern.sub(lambda m: cd[m.group(0).lower()], text)

def remove_stopwords(text):
    tokens = tokenizer.tokenize(text)
    return ' '.join([t for t in tokens if t.lower() not in stopword_list])

def clean_text(text):
    text = strip_html(text)
    text = remove_accented_chars(text)
    text = expand_contractions(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = remove_stopwords(text)
    return text.strip()

print('Cleaning reviews...')
df['cleaned_review'] = df['review'].apply(clean_text)
print('Done!')
df[['review', 'cleaned_review']].head(2)

## 4. Train / Test Split

In [ ]:
train_df = df[:4000].reset_index(drop=True)
test_df  = df[4000:].reset_index(drop=True)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

## 5. Lexicon-based Models

In [ ]:
# ── AFINN ──────────────────────────────────────────────────────────────────
afinn = Afinn()

def predict_afinn(text):
    return 1 if afinn.score(text) >= 0 else 0

test_preds_afinn = test_df['cleaned_review'].apply(predict_afinn)
print('=== AFINN ===')
print(classification_report(test_df['sentiment'], test_preds_afinn, target_names=['Negative', 'Positive']))

In [ ]:
# ── WordNet / SentiWordNet ─────────────────────────────────────────────────
def predict_wordnet(text):
    tokens = tokenizer.tokenize(text)
    scores = []
    for token in tokens:
        synsets = list(swn.senti_synsets(token))
        if synsets:
            scores.extend([s.pos_score() - s.neg_score() for s in synsets])
    return 1 if sum(scores) >= 0 else 0

test_preds_wordnet = test_df['cleaned_review'].apply(predict_wordnet)
print('=== WordNet ===')
print(classification_report(test_df['sentiment'], test_preds_wordnet, target_names=['Negative', 'Positive']))

In [ ]:
# ── VADER ──────────────────────────────────────────────────────────────────
vader = SentimentIntensityAnalyzer()

def predict_vader(text):
    return 1 if vader.polarity_scores(text)['compound'] >= 0 else 0

test_preds_vader = test_df['cleaned_review'].apply(predict_vader)
print('=== VADER ===')
print(classification_report(test_df['sentiment'], test_preds_vader, target_names=['Negative', 'Positive']))

## 6. Word2Vec Embeddings (CBOW + Skip-gram)

In [ ]:
train_tokens = [r.split() for r in train_df['cleaned_review']]
test_tokens  = [r.split() for r in test_df['cleaned_review']]

print('Training Word2Vec CBOW...')
cbow_model = gensim.models.Word2Vec(
    sentences=train_tokens, vector_size=100, window=5, min_count=1, sg=0, workers=4)

print('Training Word2Vec Skip-gram...')
skipgram_model = gensim.models.Word2Vec(
    sentences=train_tokens, vector_size=100, window=5, min_count=1, sg=1, workers=4)

# Cosine distance between similar/dissimilar words
print('\nCosine distances (CBOW):')
words = ['good', 'great', 'bad', 'terrible', 'okay']
for i in range(len(words)):
    for j in range(i+1, len(words)):
        w1, w2 = words[i], words[j]
        if w1 in cbow_model.wv and w2 in cbow_model.wv:
            d = distance.cosine(cbow_model.wv[w1], cbow_model.wv[w2])
            print(f'  {w1} ↔ {w2}: {d:.4f}')

In [ ]:
def get_review_vectors(tokens, model):
    vectors = []
    for review in tokens:
        vecs = [model.wv[w] for w in review if w in model.wv]
        vectors.append(np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size))
    return np.array(vectors)

X_train_cbow = get_review_vectors(train_tokens, cbow_model)
X_test_cbow  = get_review_vectors(test_tokens,  cbow_model)
X_train_sg   = get_review_vectors(train_tokens, skipgram_model)
X_test_sg    = get_review_vectors(test_tokens,  skipgram_model)

y_train = to_categorical(train_df['sentiment'], num_classes=2)
y_test  = to_categorical(test_df['sentiment'],  num_classes=2)

def build_nn(input_dim):
    m = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(2, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return m

print('Training NN on CBOW vectors...')
model_cbow = build_nn(100)
model_cbow.fit(X_train_cbow, y_train, epochs=5, batch_size=32,
               validation_data=(X_test_cbow, y_test), verbose=1)

print('\nTraining NN on Skip-gram vectors...')
model_sg = build_nn(100)
model_sg.fit(X_train_sg, y_train, epochs=5, batch_size=32,
             validation_data=(X_test_sg, y_test), verbose=1)

_, acc_cbow = model_cbow.evaluate(X_test_cbow, y_test, verbose=0)
_, acc_sg   = model_sg.evaluate(X_test_sg,   y_test, verbose=0)
print(f'\nCBOW + NN  accuracy: {acc_cbow:.4f}')
print(f'Skip-gram + NN accuracy: {acc_sg:.4f}')

## 7. LSTM Model

In [ ]:
# Build vocabulary
PAD_IDX, UNK_IDX = 0, 1
all_tokens = [t for review in train_tokens for t in review]
vocab = {w: i+2 for i, (w, _) in enumerate(Counter(all_tokens).items())}
MAX_LEN = 200

def encode(tokens, vocab, max_len):
    ids = [vocab.get(t, UNK_IDX) for t in tokens]
    return ids

X_train_lstm = pad_sequences(
    [encode(r, vocab, MAX_LEN) for r in train_tokens], maxlen=MAX_LEN, padding='post')
X_test_lstm  = pad_sequences(
    [encode(r, vocab, MAX_LEN) for r in test_tokens],  maxlen=MAX_LEN, padding='post')

lstm_model = Sequential([
    Embedding(input_dim=len(vocab)+2, output_dim=128, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(2, activation='softmax')
])
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lstm_model.summary()

print('Training LSTM...')
lstm_model.fit(X_train_lstm, y_train, epochs=5, batch_size=32,
               validation_data=(X_test_lstm, y_test), verbose=1)

_, acc_lstm = lstm_model.evaluate(X_test_lstm, y_test, verbose=0)
print(f'\nLSTM accuracy: {acc_lstm:.4f}')

## 8. Results Summary

In [ ]:
results = pd.DataFrame({
    'Model':    ['AFINN', 'WordNet', 'VADER', 'Word2Vec CBOW + NN', 'Word2Vec Skip-gram + NN', 'LSTM'],
    'Approach': ['Lexicon', 'Lexicon', 'Lexicon', 'Deep Learning', 'Deep Learning', 'Deep Learning'],
    'Test Accuracy': [
        accuracy_score(test_df['sentiment'], test_preds_afinn),
        accuracy_score(test_df['sentiment'], test_preds_wordnet),
        accuracy_score(test_df['sentiment'], test_preds_vader),
        acc_cbow,
        acc_sg,
        acc_lstm
    ]
})
results['Test Accuracy'] = results['Test Accuracy'].map('{:.2%}'.format)
print(results.to_string(index=False))